In [107]:
import pandas as pd

In [ ]:
df = data = pd.read_csv('
                        ')

df = df[df["position"] == "team"]

msi_2026_teams = [
    "Bilibili Gaming",
    "Top Esports",
    "Hanwha Life Esports",
    "T1",
    "G2 Esports",
    "Karmine Corp",
    "LYON",
    "Team Liquid",
    "Team Secret Whales",
    "Deep Cross Gaming",
    "FURIA"
]

international_events = [
    "MSI",
    "WLDs",
    "FST",
    "EWC"
]

df = df[
    df["league"].isin(international_events)
]

df = df[df["teamname"].isin(msi_2026_teams)]

columns = [
    "date",
    # Match information
    "league",
    "side",

    # Team information
    "teamname",
    "result",

    # Combat
    "kills",
    "deaths",
    "assists",

    # Objectives
    "dragons",
    "barons",
    "heralds",
    "towers",

    # Vision
    "visionscore",

    # Economy
    "earnedgold",
    "earned gpm",

    # Farming
    "cspm",

    # Early game
    "golddiffat15",
    "xpdiffat15",
    "csdiffat15",

    # First objectives
    "firstblood",
    "firsttower"
]

df = df[columns]

df["KDA"] = (
    (df["kills"] + df["assists"])
    / df["deaths"].replace(0, 1)
)


/var/folders/bv/z5nrz2256zd2z5klb84rgzbr0000gn/T/ipykernel_74685/631653482.py:1: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  df = data = pd.read_csv('/Users/heshuhua/Desktop/MSI 2026 Prediction/data/2026_LoL_esports_match_data_from_OraclesElixir.csv')


In [109]:
df = df.sort_values(["teamname", "date"]).reset_index(drop=True)
df["date"] = pd.to_datetime(df["date"]).dt.date

# Win rate
df["games_before"] = df.groupby("teamname").cumcount()
df["wins_before"] = (
    df.groupby("teamname")["result"]
      .cumsum()
      - df["result"]
)
df["win_rate_before"] = (
    df["wins_before"] /
    df["games_before"]
)
df["win_rate_before"] = df["win_rate_before"].fillna(0.5)

# Avergae 
features = [
    "kills",
    "deaths",
    "assists",
    "dragons",
    "barons",
    "heralds",
    "towers",
    "visionscore",
    "earnedgold",
    "earned gpm",
    "cspm",
    "golddiffat15",
    "xpdiffat15",
    "csdiffat15",
    "firstblood",
    "firsttower"
]

for feature in features:
    df[f"avg_{feature}_before"] = (
        df.groupby("teamname")[feature]
          .transform(lambda x: x.shift().expanding().mean())
    )
    df[f"avg_{feature}_before"] = (
        df[f"avg_{feature}_before"]
        .fillna(df[feature].mean())
    )
    
pre_match_columns = [
    # Match information
    "date",
    "league",
    "side",
    "teamname",
    "result",

    # Pre-match statistics
    "win_rate_before",

    "avg_kills_before",
    "avg_deaths_before",
    "avg_assists_before",

    "avg_dragons_before",
    "avg_barons_before",
    "avg_heralds_before",
    "avg_towers_before",

    "avg_visionscore_before",

    "avg_earnedgold_before",
    "avg_earned gpm_before",

    "avg_cspm_before",

    "avg_golddiffat15_before",
    "avg_xpdiffat15_before",
    "avg_csdiffat15_before",

    "avg_firstblood_before",
    "avg_firsttower_before"
]

pre_match_df = df[pre_match_columns]

pre_match_df.to_csv('2026_prematch_data.csv')